In [1]:
# Memory safety: cap this kernel to the RAM free right now so an out-of-memory
# feature build fails with a clean MemoryError instead of crashing VS Code /
# thrashing swap. This notebook builds cross-row features (lags/rolling), which
# can't be row-batched, so the guard is the protection here.
import os, sys
sys.path.insert(0, os.path.abspath("../../../Generic-Parallel-Compute-Helper/")) ; from memory_compute import *
install_memory_guard()


[memory_guard] hard cap 13.3G virtual on this kernel (total RAM 14.8G, 9.4G free now). Runaway allocations fail cleanly; bounded streaming keeps normal work well under this.


14298148864

In [2]:
import os
import sys
import numpy as np
import pandas as pd

sys.path.append(os.path.abspath("../../")) ; from EPF import variables

REGION       = variables.TARGET_REGION
REGION_CITY  = {"nsw": "sydney", "qld": "brisbane", "vic": "melbourne", "sa": "adelaide"}
CITY         = REGION_CITY[REGION]
ALL_CITIES   = list(REGION_CITY.values())
OTHER_CITIES = [c for c in ALL_CITIES if c != CITY]

PER_HOUR = 60 // variables.FEATURE_GRANULARITY_IN_MINUTES   # 12
PER_DAY  = 24 * PER_HOUR                                     # 288

# Weather is observed hourly data. The active feature blocks use only the latest
# observation available at the forecast origin, carried forward causally to the
# 5-minute grid. The optional forward-looking forecast-proxy block stays disabled.
# FCAST_24H = PER_DAY        # 288
# FCAST_48H = 2 * PER_DAY    # 576


def _wx(df: pd.DataFrame, var: str, city: str) -> pd.Series:
    """Weather variable series for a city, or NaN if that column is absent."""
    col = f"{var}_{city}"
    return df[col] if col in df.columns else pd.Series(np.nan, index=df.index)


# def _fwd_mean(s: pd.Series, w: int) -> pd.Series:
#     """Mean over the next `w` intervals (inclusive of the current one)."""
#     return s.rolling(w, min_periods=1).mean().shift(-(w - 1)).astype(np.float32)


# def _fwd_sum(s: pd.Series, w: int) -> pd.Series:
#     """Sum over the next `w` intervals (inclusive of the current one)."""
#     return s.rolling(w, min_periods=1).sum().shift(-(w - 1)).astype(np.float32)

In [3]:
df = read_parquet_float32("../1_Dataset/Processed_data/5_weather.parquet")

# Compatibility guard for processed weather files created before causal
# resampling was introduced. Exact-hour rows contain the original observations;
# rebuild every intra-hour row from the latest exact-hour value so this notebook
# cannot consume the old linear interpolation between current and future hours.
weather_index = df.index
hourly_observations = df.loc[(weather_index.minute == 0) & (weather_index.second == 0)]
df = hourly_observations.reindex(weather_index).ffill()
del hourly_observations, weather_index

df_core_columns = df.columns
df_base = df
df_base[:10]

Loading..:   0%|          | 0/10 [00:00<?, ?batch/s]

Loading..: 100%|██████████| 10/10 [00:00<00:00, 11.88batch/s]


,temp_sydney,feelslike_sydney,dew_sydney,humidity_sydney,precip_sydney,precipprob_sydney,snow_sydney,snowdepth_sydney,windgust_sydney,windspeed_sydney,...,windspeed_adelaide,winddir_adelaide,sealevelpressure_adelaide,cloudcover_adelaide,visibility_adelaide,solarradiation_adelaide,solarenergy_adelaide,uvindex_adelaide,severerisk_adelaide,stations_adelaide
SETTLEMENTDATE,,,,,,,,,,,,,,,,,,,,,
2018-01-01 00:00:00,22.400000,22.400000,18.299999,77.779999,0.0,0.0,0.0,0.0,28.799999,1.700,...,8.100,127.00,1010.799988,6.700000,10.0,0.0,0.0,0.0,NaN,NaN
2018-01-01 00:05:00,22.383333,22.383333,18.316668,77.930832,0.0,0.0,0.0,0.0,28.441668,1.675,...,7.875,124.75,1010.775024,8.225000,10.0,0.0,0.0,0.0,NaN,NaN
2018-01-01 00:10:00,22.366667,22.366667,18.333334,78.081665,0.0,0.0,0.0,0.0,28.083334,1.650,...,7.650,122.50,1010.750000,9.750000,10.0,0.0,0.0,0.0,NaN,NaN
2018-01-01 00:15:00,22.350000,22.350000,18.350000,78.232498,0.0,0.0,0.0,0.0,27.725000,1.625,...,7.425,120.25,1010.724976,11.275000,10.0,0.0,0.0,0.0,NaN,NaN
2018-01-01 00:20:00,22.333334,22.333334,18.366667,78.383331,0.0,0.0,0.0,0.0,27.366667,1.600,...,7.200,118.00,1010.700012,12.800000,10.0,0.0,0.0,0.0,NaN,NaN
2018-01-01 00:25:00,22.316668,22.316668,18.383333,78.534164,0.0,0.0,0.0,0.0,27.008333,1.575,...,6.975,115.75,1010.674988,14.325000,10.0,0.0,0.0,0.0,NaN,NaN
2018-01-01 00:30:00,22.299999,22.299999,18.400000,78.684998,0.0,0.0,0.0,0.0,26.650000,1.550,...,6.750,113.50,1010.650024,15.850000,10.0,0.0,0.0,0.0,NaN,NaN
2018-01-01 00:35:00,22.283333,22.283333,18.416666,78.835831,0.0,0.0,0.0,0.0,26.291666,1.525,...,6.525,111.25,1010.625000,17.375000,10.0,0.0,0.0,0.0,NaN,NaN
2018-01-01 00:40:00,22.266666,22.266666,18.433332,78.986664,0.0,0.0,0.0,0.0,25.933332,1.500,...,6.300,109.00,1010.599976,18.900000,10.0,0.0,0.0,0.0,NaN,NaN


In [4]:
def _add_weather_now_features(df: pd.DataFrame) -> pd.DataFrame:
    """
    Current-interval weather for the target region's reference city plus derived
    cooling/heating degrees. Hot afternoons drive air-conditioning demand while
    cold snaps drive heating; both lift price. Every value is the latest hourly
    observation available by the interval, so no later observation is used.
    Returns only the new columns to avoid copying the full base frame.
    """
    C = CITY
    t = _wx(df, "temp", C)
    new_cols = {}
    new_cols[f"temp_{C}_now"]       = t.astype(np.float32)
    new_cols[f"cdd_{C}"]            = (t - 18).clip(lower=0).astype(np.float32)
    new_cols[f"hdd_{C}"]            = (18 - t).clip(lower=0).astype(np.float32)
    new_cols[f"feelslike_{C}_now"]  = _wx(df, "feelslike", C).astype(np.float32)
    new_cols[f"humidity_{C}_now"]   = _wx(df, "humidity", C).astype(np.float32)
    new_cols[f"windspeed_{C}_now"]  = _wx(df, "windspeed", C).astype(np.float32)
    new_cols[f"windgust_{C}_now"]   = _wx(df, "windgust", C).astype(np.float32)
    new_cols[f"solarrad_{C}_now"]   = _wx(df, "solarradiation", C).astype(np.float32)
    new_cols[f"cloudcover_{C}_now"] = _wx(df, "cloudcover", C).astype(np.float32)
    return pd.DataFrame(new_cols, index=df.index)


new_df = _add_weather_now_features(df_base)
df = pd.concat([df, new_df], axis=1)
new_df[:10]


,temp_sydney_now,cdd_sydney,hdd_sydney,feelslike_sydney_now,humidity_sydney_now,windspeed_sydney_now,windgust_sydney_now,solarrad_sydney_now,cloudcover_sydney_now
SETTLEMENTDATE,,,,,,,,,
2018-01-01 00:00:00,22.400000,4.400000,0.0,22.400000,77.779999,1.700,28.799999,0.0,91.599998
2018-01-01 00:05:00,22.383333,4.383333,0.0,22.383333,77.930832,1.675,28.441668,0.0,92.025002
2018-01-01 00:10:00,22.366667,4.366667,0.0,22.366667,78.081665,1.650,28.083334,0.0,92.449997
2018-01-01 00:15:00,22.350000,4.350000,0.0,22.350000,78.232498,1.625,27.725000,0.0,92.875000
2018-01-01 00:20:00,22.333334,4.333334,0.0,22.333334,78.383331,1.600,27.366667,0.0,93.300003
2018-01-01 00:25:00,22.316668,4.316668,0.0,22.316668,78.534164,1.575,27.008333,0.0,93.724998
2018-01-01 00:30:00,22.299999,4.299999,0.0,22.299999,78.684998,1.550,26.650000,0.0,94.150002
2018-01-01 00:35:00,22.283333,4.283333,0.0,22.283333,78.835831,1.525,26.291666,0.0,94.574997
2018-01-01 00:40:00,22.266666,4.266666,0.0,22.266666,78.986664,1.500,25.933332,0.0,95.000000


In [5]:
def _add_weather_history_features(df: pd.DataFrame) -> pd.DataFrame:
    """
    Backward-looking weather regime for the target city: temperature lags, daily
    rolling level/extremes, accumulated cooling/heating degree-hours and recent
    wind. Captures heat build-up and cold spells. Look-back only, no leakage.
    Returns only the new columns to avoid copying the full base frame.
    """
    C = CITY
    t = _wx(df, "temp", C)
    cdd = (t - 18).clip(lower=0)
    hdd = (18 - t).clip(lower=0)
    new_cols = {}
    for lag in [PER_HOUR, PER_DAY]:
        new_cols[f"temp_{C}_lag_{lag}"] = t.shift(lag).astype(np.float32)
    new_cols[f"temp_{C}_rmean_{PER_DAY}"] = t.rolling(PER_DAY, min_periods=PER_HOUR).mean().astype(np.float32)
    new_cols[f"temp_{C}_rmax_{PER_DAY}"]  = t.rolling(PER_DAY, min_periods=PER_HOUR).max().astype(np.float32)
    new_cols[f"temp_{C}_rmin_{PER_DAY}"]  = t.rolling(PER_DAY, min_periods=PER_HOUR).min().astype(np.float32)
    new_cols[f"cdd_{C}_rsum_24h"] = (cdd.rolling(PER_DAY, min_periods=PER_HOUR).sum() / PER_HOUR).astype(np.float32)
    new_cols[f"hdd_{C}_rsum_24h"] = (hdd.rolling(PER_DAY, min_periods=PER_HOUR).sum() / PER_HOUR).astype(np.float32)
    new_cols[f"windspeed_{C}_rmean_{PER_DAY}"] = (
        _wx(df, "windspeed", C).rolling(PER_DAY, min_periods=PER_HOUR).mean().astype(np.float32)
    )
    return pd.DataFrame(new_cols, index=df.index)


new_df = _add_weather_history_features(df_base)
df = pd.concat([df, new_df], axis=1)
new_df[:10]


,temp_sydney_lag_12,temp_sydney_lag_288,temp_sydney_rmean_288,temp_sydney_rmax_288,temp_sydney_rmin_288,cdd_sydney_rsum_24h,hdd_sydney_rsum_24h,windspeed_sydney_rmean_288
SETTLEMENTDATE,,,,,,,,
2018-01-01 00:00:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2018-01-01 00:05:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2018-01-01 00:10:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2018-01-01 00:15:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2018-01-01 00:20:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2018-01-01 00:25:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2018-01-01 00:30:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2018-01-01 00:35:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2018-01-01 00:40:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [6]:
# def _add_weather_forecast_features(df: pd.DataFrame) -> pd.DataFrame:
#     """
#     Forward-looking weather over the next 24h / 48h for the target city, used as
#     a perfect-foresight proxy for the weather *forecast* a real operator would
#     hold at run time (the model's price horizon is 48h). Expected temperature,
#     cooling/heating degree-hours, solar irradiance and wind over the horizon are
#     the primary drivers of demand and variable-renewable output. Tagged "_fcast".
#     """
#     df = df.copy()
#     C = CITY
#     t = _wx(df, "temp", C)
#     cdd = (t - 18).clip(lower=0)
#     hdd = (18 - t).clip(lower=0)
#     solar = _wx(df, "solarradiation", C)
#     wind = _wx(df, "windspeed", C)
#     new_cols = {}
#     for w, lab in [(FCAST_24H, "24h"), (FCAST_48H, "48h")]:
#         new_cols[f"temp_{C}_fcast_mean_{lab}"]      = _fwd_mean(t, w)
#         new_cols[f"temp_{C}_fcast_max_{lab}"]       = t.rolling(w, min_periods=1).max().shift(-(w - 1)).astype(np.float32)
#         new_cols[f"cdd_{C}_fcast_sum_{lab}"]        = (_fwd_sum(cdd, w) / PER_HOUR).astype(np.float32)
#         new_cols[f"hdd_{C}_fcast_sum_{lab}"]        = (_fwd_sum(hdd, w) / PER_HOUR).astype(np.float32)
#         new_cols[f"solarrad_{C}_fcast_mean_{lab}"]  = _fwd_mean(solar, w)
#         new_cols[f"windspeed_{C}_fcast_mean_{lab}"] = _fwd_mean(wind, w)
#     return pd.concat([df, pd.DataFrame(new_cols, index=df.index)], axis=1)


# new_df = _add_weather_forecast_features(df_base)
# df = pd.concat([df, new_df.drop(columns=df_core_columns)], axis=1)
# new_df[:10]

In [7]:
def _add_cross_city_weather_features(df: pd.DataFrame) -> pd.DataFrame:
    """
    National weather context and the other capital cities' conditions.
    Coincident heat across the NEM lifts demand everywhere and correlates VRE
    output, tightening the interconnected market. Latest-known values only.
    Returns only the new columns to avoid copying the full base frame.
    """
    new_cols = {}
    nat_temp = pd.concat([_wx(df, "temp", c) for c in ALL_CITIES], axis=1).mean(axis=1)
    new_cols["nat_temp_mean"] = nat_temp.astype(np.float32)
    new_cols["nat_cdd_mean"]  = (nat_temp - 18).clip(lower=0).astype(np.float32)
    new_cols["nat_hdd_mean"]  = (18 - nat_temp).clip(lower=0).astype(np.float32)
    for c in OTHER_CITIES:
        new_cols[f"temp_{c}_now"]      = _wx(df, "temp", c).astype(np.float32)
        new_cols[f"solarrad_{c}_now"]  = _wx(df, "solarradiation", c).astype(np.float32)
        new_cols[f"windspeed_{c}_now"] = _wx(df, "windspeed", c).astype(np.float32)
    return pd.DataFrame(new_cols, index=df.index)


new_df = _add_cross_city_weather_features(df_base)
df = pd.concat([df, new_df], axis=1)
new_df[:10]


,nat_temp_mean,nat_cdd_mean,nat_hdd_mean,temp_brisbane_now,solarrad_brisbane_now,windspeed_brisbane_now,temp_melbourne_now,solarrad_melbourne_now,windspeed_melbourne_now,temp_adelaide_now,solarrad_adelaide_now,windspeed_adelaide_now
SETTLEMENTDATE,,,,,,,,,,,,
2018-01-01 00:00:00,20.150000,2.150000,0.0,25.400000,0.0,1.900000,17.400000,0.0,6.400000,15.400000,0.0,8.100
2018-01-01 00:05:00,20.181252,2.181252,0.0,25.433332,0.0,1.741667,17.358334,0.0,6.333333,15.550000,0.0,7.875
2018-01-01 00:10:00,20.212500,2.212500,0.0,25.466667,0.0,1.583333,17.316668,0.0,6.266667,15.700000,0.0,7.650
2018-01-01 00:15:00,20.243750,2.243750,0.0,25.500000,0.0,1.425000,17.275000,0.0,6.200000,15.850000,0.0,7.425
2018-01-01 00:20:00,20.275002,2.275002,0.0,25.533333,0.0,1.266667,17.233334,0.0,6.133333,16.000000,0.0,7.200
2018-01-01 00:25:00,20.306252,2.306252,0.0,25.566668,0.0,1.108333,17.191668,0.0,6.066667,16.150000,0.0,6.975
2018-01-01 00:30:00,20.337502,2.337502,0.0,25.600000,0.0,0.950000,17.150000,0.0,6.000000,16.299999,0.0,6.750
2018-01-01 00:35:00,20.368748,2.368748,0.0,25.633333,0.0,0.791667,17.108334,0.0,5.933333,16.450001,0.0,6.525
2018-01-01 00:40:00,20.400000,2.400000,0.0,25.666666,0.0,0.633333,17.066668,0.0,5.866667,16.600000,0.0,6.300


In [8]:
# Retain core columns: each 5-minute row holds only the latest hourly observation
# available at t. The forward-looking _fcast block stays disabled.
print("Total features:", df.shape[1])
df.to_parquet("../2_Features_build/Feature_data/5_weather.parquet")
df.shape

Total features: 104


(902581, 104)

In [9]:
# Free this kernel's memory so the next notebook has RAM to work with
# (clears data variables + returns freed heap to the OS).
release_memory()


[release_memory] cleared 16 variable(s); kernel rss 0.52G, 8.9G RAM free now
